# 01 波形基础与合成

内容脉络：
1. 环境自检与统一配置
2. 合成基础波形（正弦 / 方波 / 锯齿 / 三角）
3. 合成噪声（白噪声 / 粉红噪声）
4. 采样定理与混叠效应
5. 量化与位深
6. 响度归一化、削波与相位
7. 加载真实录音并可视化

合成素材保存至 `CODE/chapter05/output_audio/`，供后续 Notebook 复用。

## 1. 环境自检与配置

In [ ]:
import sys
assert sys.version_info >= (3, 9), "需要 Python 3.9+"

import numpy as np
import matplotlib.pyplot as plt

# 配置 matplotlib 中文字体（macOS/Windows/Linux）
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['PingFang HK', 'STHeiti', 'Heiti TC', 'Arial Unicode MS', 'Hiragino Sans GB', 'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False  # 解决负号显示为方框的问题

from IPython import display as ipydisplay
from pathlib import Path
import warnings

# 音频核心库
try:
    import librosa
    import librosa.display
    print(f"librosa={librosa.__version__}")
except ImportError as e:
    raise ImportError(f"缺少 librosa：{e}，请运行 pip install -r requirements.txt")

# scipy 用于估计噪声的 periodogram
try:
    from scipy import signal as scipy_signal
except ImportError as e:
    raise ImportError(f"缺少 scipy：{e}，请运行 pip install -r requirements.txt")

# 全局参数：统一使用 22050 Hz 单声道，与本章一致
SAMPLE_RATE = 22050

# 路径推断：从 cwd 向上找含 CODE/datasets 的目录
_p = Path.cwd()
while not (_p / "CODE" / "datasets").exists():
    _parent = _p.parent
    if _parent == _p:
        raise FileNotFoundError("未找到项目根目录（包含 CODE/datasets 的目录），请在项目内运行本 Notebook")
    _p = _parent
BASE_DIR = _p
DATASET_DIR = BASE_DIR / "CODE" / "datasets" / "audio_author"
OUTPUT_AUDIO_DIR = BASE_DIR / "CODE" / "chapter05" / "output_audio"
OUTPUT_FIG_DIR = BASE_DIR / "CODE" / "chapter05" / "output_figures"
OUTPUT_AUDIO_DIR.mkdir(exist_ok=True)
OUTPUT_FIG_DIR.mkdir(exist_ok=True)

print(f"数据集目录：{DATASET_DIR.resolve()}")
print(f"音频输出目录：{OUTPUT_AUDIO_DIR.resolve()}")

In [ ]:
# 全局绘图风格与可复现随机数生成器
plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.figsize"] = (10, 3)
RNG = np.random.default_rng(20260805)


## 2. 合成基础波形

音乐与声学中常用纯音和周期波形说明谐波结构。
- 正弦波（sine）：理想纯音，只有一个频率成分
- 方波（square）：理想形式只含奇次谐波
- 锯齿波（sawtooth）：理想形式同时含奇次与偶次谐波
- 三角波（triangle）：理想形式只含奇次谐波，幅度衰减更快

离散系统不能无混叠地容纳无限多谐波。下面用加法合成，只保留低于 Nyquist 频率的项，因此得到的是这些理想波形的**带限近似**。

In [ ]:
def _time_axis(frequency_hz, duration_sec, sample_rate):
    if frequency_hz <= 0 or duration_sec < 0 or sample_rate <= 0:
        raise ValueError("frequency_hz、sample_rate 必须为正，duration_sec 不能为负")
    return np.arange(int(sample_rate * duration_sec), dtype=float) / sample_rate

def _peak_normalize(samples, amplitude):
    if amplitude < 0:
        raise ValueError("amplitude 不能为负")
    peak = np.max(np.abs(samples)) if samples.size else 0.0
    return samples if peak == 0 else amplitude * samples / peak

def _harmonic_numbers_below_nyquist(frequency_hz, sample_rate, odd_only=False):
    """返回严格低于 Nyquist 频率的正整数谐波序号。"""
    stop = int(np.ceil(sample_rate / (2 * frequency_hz)))
    return np.arange(1, stop, 2 if odd_only else 1)

def generate_sine_wave(frequency_hz, duration_sec, sample_rate=SAMPLE_RATE, amplitude=0.5):
    """生成正弦波。"""
    time_axis = _time_axis(frequency_hz, duration_sec, sample_rate)
    return time_axis, amplitude * np.sin(2 * np.pi * frequency_hz * time_axis)

def generate_square_wave(frequency_hz, duration_sec, sample_rate=SAMPLE_RATE, amplitude=0.5):
    """用低于 Nyquist 的奇次谐波生成带限方波近似。"""
    time_axis = _time_axis(frequency_hz, duration_sec, sample_rate)
    harmonic_numbers = _harmonic_numbers_below_nyquist(frequency_hz, sample_rate, odd_only=True)
    wave_samples = np.sum(
        np.sin(2 * np.pi * frequency_hz * harmonic_numbers[:, None] * time_axis)
        / harmonic_numbers[:, None], axis=0
    ) if harmonic_numbers.size else np.zeros_like(time_axis)
    return time_axis, _peak_normalize(wave_samples, amplitude)

def generate_sawtooth_wave(frequency_hz, duration_sec, sample_rate=SAMPLE_RATE, amplitude=0.5):
    """用低于 Nyquist 的全部整数谐波生成带限锯齿波近似。"""
    time_axis = _time_axis(frequency_hz, duration_sec, sample_rate)
    harmonic_numbers = _harmonic_numbers_below_nyquist(frequency_hz, sample_rate)
    signs = (-1.0) ** (harmonic_numbers + 1)
    wave_samples = np.sum(
        signs[:, None] * np.sin(2 * np.pi * frequency_hz * harmonic_numbers[:, None] * time_axis)
        / harmonic_numbers[:, None], axis=0
    ) if harmonic_numbers.size else np.zeros_like(time_axis)
    return time_axis, _peak_normalize(wave_samples, amplitude)

def generate_triangle_wave(frequency_hz, duration_sec, sample_rate=SAMPLE_RATE, amplitude=0.5):
    """用低于 Nyquist 的奇次谐波生成带限三角波近似。"""
    time_axis = _time_axis(frequency_hz, duration_sec, sample_rate)
    harmonic_numbers = _harmonic_numbers_below_nyquist(frequency_hz, sample_rate, odd_only=True)
    signs = (-1.0) ** ((harmonic_numbers - 1) // 2)
    wave_samples = np.sum(
        signs[:, None] * np.sin(2 * np.pi * frequency_hz * harmonic_numbers[:, None] * time_axis)
        / harmonic_numbers[:, None] ** 2, axis=0
    ) if harmonic_numbers.size else np.zeros_like(time_axis)
    return time_axis, _peak_normalize(wave_samples, amplitude)

In [ ]:
# 统一参数：A4 = 440 Hz，时长 2 秒，振幅 0.5（避免削波）
PITCH_A4 = 440.0
DURATION_SHORT = 2.0
AMPLITUDE_DEFAULT = 0.5

time_axis, sine_samples = generate_sine_wave(PITCH_A4, DURATION_SHORT, amplitude=AMPLITUDE_DEFAULT)
_, square_samples = generate_square_wave(PITCH_A4, DURATION_SHORT, amplitude=AMPLITUDE_DEFAULT)
_, sawtooth_samples = generate_sawtooth_wave(PITCH_A4, DURATION_SHORT, amplitude=AMPLITUDE_DEFAULT)
_, triangle_samples = generate_triangle_wave(PITCH_A4, DURATION_SHORT, amplitude=AMPLITUDE_DEFAULT)

# 保存合成波形到 output_audio/，供后续 Notebook 使用
def save_audio(filename, samples, sample_rate=SAMPLE_RATE):
    """保存为 16-bit PCM WAV"""
    path = OUTPUT_AUDIO_DIR / filename
    # 确保不削波：限制到 [-1, 1] 后再转 int16
    samples_clipped = np.clip(samples, -1.0, 1.0)
    samples_int16 = (samples_clipped * 32767).astype(np.int16)
    # 使用 scipy.io.wavfile 或 soundfile，这里优先 soundfile，回退 wavfile
    try:
        import soundfile as sf
        sf.write(path, samples_clipped, sample_rate, subtype="PCM_16")
    except ImportError:
        from scipy.io import wavfile
        wavfile.write(path, sample_rate, samples_int16)
    return path

save_audio("sine_440hz.wav", sine_samples)
save_audio("square_440hz.wav", square_samples)
save_audio("sawtooth_440hz.wav", sawtooth_samples)
save_audio("triangle_440hz.wav", triangle_samples)
print("已保存四种基础波形到 output_audio/")

In [ ]:
# 可视化：四种波形的前 10 ms（约 220 个采样点 @ 22.05k）
n_show = int(SAMPLE_RATE * 0.01)  # 10 ms
waveforms = {
    "正弦波": sine_samples,
    "方波": square_samples,
    "锯齿波": sawtooth_samples,
    "三角波": triangle_samples,
}

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, sharey=True)
axes = axes.flatten()
for ax, (name, samples) in zip(axes, waveforms.items()):
    ax.plot(time_axis[:n_show], samples[:n_show], lw=1.2, color="black")
    ax.set_title(name)
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("振幅")
    ax.set_ylim(-0.6, 0.6)
    ax.axhline(0, color="gray", ls="--", lw=0.5)
fig.suptitle("基础波形 @ 440 Hz（前 10 ms）", fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "basic_waveforms.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
print("正弦波（纯音）：")
display(ipydisplay.Audio(sine_samples, rate=SAMPLE_RATE))

In [ ]:
print("锯齿波（带限谐波近似）：")
display(ipydisplay.Audio(sawtooth_samples, rate=SAMPLE_RATE))

In [ ]:
print("方波（带限奇次谐波近似）：")
display(ipydisplay.Audio(square_samples, rate=SAMPLE_RATE))

### 频谱对比：为什么音色不同？

正弦波只有基频，其余波形含有不同组合的谐波。
用 FFT 比较各波形的频谱结构。

In [ ]:
def plot_spectrum_comparison(samples_dict, sample_rate=SAMPLE_RATE, max_freq_hz=5000):
    """绘制各波形幅度谱（取单帧 FFT）"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 6))
    axes = axes.flatten()
    n_fft = 8192
    freq_axis = np.fft.rfftfreq(n_fft, 1 / sample_rate)
    for ax, (name, samples) in zip(axes, samples_dict.items()):
        # 取前 n_fft 个采样做 FFT，加 Hann 窗减少泄漏
        frame = samples[:n_fft] * np.hanning(n_fft)
        magnitude = np.abs(np.fft.rfft(frame))
        # 转 dB，参考值为 FFT 最大幅值
        magnitude_db = 20 * np.log10(magnitude / np.max(magnitude) + 1e-10)
        ax.plot(freq_axis, magnitude_db, lw=1.0, color="0.4")
        ax.set_xlim(0, max_freq_hz)
        ax.set_ylim(-60, 5)
        ax.set_title(name)
        ax.set_xlabel("频率 (Hz)")
        ax.set_ylabel("幅度 (dB)")
        # 标出谐波位置
        for harmonic in range(1, int(max_freq_hz / PITCH_A4) + 1):
            ax.axvline(harmonic * PITCH_A4, color="red", ls="--", lw=0.4, alpha=0.6)
    fig.suptitle("基础波形的谐波结构（dB 尺度）", fontsize=13)
    plt.tight_layout()
    plt.savefig(OUTPUT_FIG_DIR / "waveform_spectra.png", dpi=600, bbox_inches="tight")
    plt.show()

plot_spectrum_comparison(waveforms)

## 3. 合成噪声

噪声可作为比较周期结构与谐波结构的参考信号。
- 白噪声：理论功率谱密度为常数；有限样本的 periodogram 会围绕该期望随机起伏
- 粉红噪声：理论上每倍频程包含相等能量，功率谱密度随频率近似按 $1/f$ 衰减

In [ ]:
def generate_white_noise(duration_sec, sample_rate=SAMPLE_RATE, amplitude=0.3, rng=RNG):
    """生成高斯白噪声样本，并整体峰值归一化；PSD 期望形状保持为常数。"""
    num_samples = int(sample_rate * duration_sec)
    white = rng.normal(size=num_samples)
    return _peak_normalize(white, amplitude)

def generate_pink_noise(duration_sec, sample_rate=SAMPLE_RATE, amplitude=0.3, rng=RNG):
    """粉红噪声近似：期望 PSD 与 1/f 成正比（频域整形法）。"""
    num_samples = int(sample_rate * duration_sec)
    white = rng.normal(size=num_samples)
    spectrum = np.fft.rfft(white)
    frequencies = np.fft.rfftfreq(num_samples, d=1 / sample_rate)
    pink_filter = np.zeros_like(frequencies)
    positive = frequencies > 0
    pink_filter[positive] = 1.0 / np.sqrt(frequencies[positive])
    spectrum_filtered = spectrum * pink_filter
    pink = np.fft.irfft(spectrum_filtered, n=num_samples)
    peak = np.max(np.abs(pink))
    return amplitude * pink / peak if peak > 0 else pink

DURATION_NOISE = 3.0
white_noise_samples = generate_white_noise(DURATION_NOISE, amplitude=0.3)
pink_noise_samples = generate_pink_noise(DURATION_NOISE, amplitude=0.3)

save_audio("white_noise.wav", white_noise_samples)
save_audio("pink_noise.wav", pink_noise_samples)
print("已保存白噪声与粉红噪声（固定随机种子）")


In [ ]:
# 噪声波形与 periodogram 对比；有限样本曲线会围绕理论期望随机起伏
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
time_axis_noise = np.arange(len(white_noise_samples)) / SAMPLE_RATE
n_show_noise = int(SAMPLE_RATE * 0.05)  # 50 ms

axes[0, 0].plot(time_axis_noise[:n_show_noise], white_noise_samples[:n_show_noise], color="0.25", lw=0.8)
axes[0, 0].set_title("白噪声（时域）")
axes[0, 0].set_xlabel("时间 (s)")
axes[0, 0].set_ylabel("振幅")

axes[0, 1].plot(time_axis_noise[:n_show_noise], pink_noise_samples[:n_show_noise], color="0.4", lw=0.8)
axes[0, 1].set_title("粉红噪声（时域）")
axes[0, 1].set_xlabel("时间 (s)")
axes[0, 1].set_ylabel("振幅")

def plot_psd(ax, samples, sample_rate, title, color="0.25"):
    freq, psd = scipy_signal.periodogram(samples, fs=sample_rate, window="hann")
    ax.semilogy(freq, psd, color=color, lw=0.8)
    ax.set_xlim(20, sample_rate // 2)
    ax.set_xlabel("频率 (Hz)")
    ax.set_ylabel("功率谱密度")
    ax.set_title(title)
    return freq, psd

freq_white, psd_white = plot_psd(
    axes[1, 0], white_noise_samples, SAMPLE_RATE, "白噪声 periodogram"
)
freq_pink, psd_pink = plot_psd(
    axes[1, 1], pink_noise_samples, SAMPLE_RATE, "粉红噪声 periodogram", color="0.4"
)

white_ref = np.mean(psd_white[(freq_white >= 100) & (freq_white <= 10000)])
axes[1, 0].axhline(white_ref, color="0.65", ls="--", lw=1.2, label="常数期望（示意）")
axes[1, 0].legend()

f_ref = np.array([100, 10000])
idx_1k = np.argmin(np.abs(freq_pink - 1000))
y_ref = psd_pink[idx_1k] * (1000 / f_ref)
axes[1, 1].plot(f_ref, y_ref, color="0.65", ls="--", lw=1.2, label="1/f 期望斜率（示意）")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "noise_waveforms_psd.png", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:
print("白噪声（期望功率谱密度为常数；有限样本会随机起伏）：")
display(ipydisplay.Audio(white_noise_samples, rate=SAMPLE_RATE))


In [ ]:
print("粉红噪声（期望功率谱密度近似 1/f，低频能量更强）：")
display(ipydisplay.Audio(pink_noise_samples, rate=SAMPLE_RATE))


## 4. 采样定理与混叠效应

Nyquist-Shannon 采样定理要求带限信号的最高频率严格低于采样率的一半，也常写作 $f_s \geq 2f_{max}$，但临界频率处不能含有需要保留的分量。

如果采样率不足，高频成分会与较低频率产生相同的采样序列，这就是 **混叠（aliasing）**。

下面用 3 Hz 余弦波演示：
- 10 Hz：满足 $10 > 2 \times 3$，采样点足以支持带限重建
- 4 Hz：不满足 $4 > 2 \times 3$，3 Hz 与 1 Hz 余弦在这些采样时刻取值相同


In [ ]:
# 用高采样率曲线近似展示连续时间信号
SR_CONTINUOUS = 1000  # Hz
DURATION_ALIAS = 1.0  # second
FREQ_SIGNAL = 3.0     # Hz

time_continuous = np.linspace(0, DURATION_ALIAS, int(SR_CONTINUOUS * DURATION_ALIAS), endpoint=False)
signal_continuous = np.cos(2 * np.pi * FREQ_SIGNAL * time_continuous)

sampling_rates = [10, 4]
sampled_data = {}
for sample_rate_demo in sampling_rates:
    sample_times = np.arange(0, DURATION_ALIAS, 1 / sample_rate_demo)
    sample_values = np.cos(2 * np.pi * FREQ_SIGNAL * sample_times)
    sampled_data[sample_rate_demo] = (sample_times, sample_values)

ALIAS_FREQ = abs(FREQ_SIGNAL - 4)
time_alias = time_continuous
signal_alias = np.cos(2 * np.pi * ALIAS_FREQ * time_alias)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5), sharey=True)

axes[0].plot(time_continuous, signal_continuous, color="0.5", lw=1.2, label="连续时间参考")
axes[0].set_title("3 Hz 余弦")
axes[0].set_xlabel("时间 (s)")
axes[0].set_ylabel("振幅")
axes[0].legend(fontsize=8)
axes[0].set_xlim(0, DURATION_ALIAS)

axes[1].plot(time_continuous, signal_continuous, color="0.5", lw=0.8, alpha=0.5)
t_s, s_s = sampled_data[10]
stems1 = axes[1].stem(t_s, s_s, basefmt=" ")
plt.setp(stems1.markerline, color="0.25", markersize=5)
plt.setp(stems1.stemlines, color="0.25", linewidth=1.0)
axes[1].set_title(r"采样 @ 10 Hz（$f_s > 2f_{max}$）")
axes[1].set_xlabel("时间 (s)")
axes[1].set_xlim(0, DURATION_ALIAS)

axes[2].plot(time_continuous, signal_continuous, color="0.5", lw=0.8, alpha=0.5, label="原始 3 Hz")
t_s, s_s = sampled_data[4]
stems2 = axes[2].stem(t_s, s_s, basefmt=" ")
plt.setp(stems2.markerline, color="0.25", markersize=5, marker="s")
plt.setp(stems2.stemlines, color="0.25", linewidth=1.0)
axes[2].plot(time_alias, signal_alias, color="0.2", ls="--", lw=1.5, alpha=0.8, label="混叠 1 Hz")
axes[2].set_title(r"采样 @ 4 Hz（$f_s < 2f_{max}$）")
axes[2].set_xlabel("时间 (s)")
axes[2].set_xlim(0, DURATION_ALIAS)
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "aliasing_demo.png", dpi=600, bbox_inches="tight")
plt.show()


右图中的 4 个采样点同时落在 3 Hz 与 1 Hz 余弦曲线上。仅凭这组离散样本无法区分二者；在假设信号带限于 Nyquist 频率以下时，重建会选择 1 Hz 分量。


## 5. 量化与位深

采样解决了「何时取点」，量化解决「每个点记多少级」。
位深 $b$ 决定量化等级数 $2^b$。对不过载、无抖动的理想均匀量化器和满量程正弦，常用近似为 $\mathrm{SNR} \approx 6.02b + 1.76$ dB；它不是任意音频的固定上限或实测值。

下面在同一段钢琴录音上，模拟 16-bit、8-bit、4-bit 量化的听感差异。

In [ ]:
# 加载钢琴独奏，取前 5 秒
piano_path = DATASET_DIR / "piano_solo.wav"
if not piano_path.exists():
    raise FileNotFoundError(f"未找到钢琴音频：{piano_path}。请确认 datasets/audio_author/ 路径正确。")

piano_samples, piano_sr = librosa.load(piano_path, sr=SAMPLE_RATE, mono=True, duration=5.0)
print(f"加载钢琴音频：采样率 {piano_sr} Hz，时长 {len(piano_samples)/piano_sr:.2f} s，共 {len(piano_samples)} 个采样点")

In [ ]:
def quantize_float_to_bitdepth(samples, bit_depth):
    """用标准有符号均匀 PCM 码本量化 [-1, 1) 浮点样本。"""
    scale = 2 ** (bit_depth - 1)
    integer_codes = np.round(np.asarray(samples) * scale)
    integer_codes = np.clip(integer_codes, -scale, scale - 1)
    return integer_codes / scale

def compute_snr(original, quantized):
    """计算信号与量化误差的信噪比（dB）。"""
    noise = original - quantized
    signal_power = np.mean(original ** 2)
    noise_power = np.mean(noise ** 2)
    return np.inf if noise_power == 0 else 10 * np.log10(signal_power / noise_power)

bit_depths = [16, 8, 4]
quantized_versions = {}
for bits in bit_depths:
    q = quantize_float_to_bitdepth(piano_samples, bits)
    quantized_versions[bits] = q
    snr = compute_snr(piano_samples, q)
    theory_snr = 6.02 * bits + 1.76
    print(f"{bits:2d}-bit: 实测 SNR = {snr:6.2f} dB | 满量程正弦理论值 ≈ {theory_snr:5.1f} dB")


In [ ]:
# 可视化：约一个周期内的原始波形与三种量化波形
t_demo = np.linspace(0, 1, SAMPLE_RATE, endpoint=False)
sine_demo = 0.9 * np.sin(2 * np.pi * 440 * t_demo)
period_samples = int(SAMPLE_RATE / 440)       # 约 1 周期（2.27 ms），取 50 点
t_slice = t_demo[:period_samples]
orig_slice = sine_demo[:period_samples]

fig, axes = plt.subplots(2, 2, figsize=(11, 6))

# 左上为原始正弦波（光滑曲线，作为参照）
axes[0, 0].plot(t_slice, orig_slice, color="0.15", lw=1.2, label="原始")
axes[0, 0].set_title("原始正弦波（约 1 周期）")
axes[0, 0].set_ylabel("振幅")
axes[0, 0].set_ylim(-1.0, 1.0)
axes[0, 0].legend(loc="lower left", fontsize=8)
axes[0, 0].axhline(0, color="0.6", ls="--", lw=0.5)

for idx, bits in enumerate(bit_depths):
    row = (idx + 1) // 2
    col = (idx + 1) % 2
    ax = axes[row, col]
    q = quantize_float_to_bitdepth(sine_demo, bits)
    q_step = 2.0 / (2 ** bits)

    # 原始：浅灰细线
    ax.plot(t_slice, orig_slice, color="0.6", lw=0.8, alpha=0.7, label="原始")
    # 量化后：深黑阶梯
    ax.step(t_slice, q[:period_samples], where="mid", color="0.15", lw=1.2, label=f"{bits}-bit")

    ax.set_title(f"{bits}-bit | 量化级数 = {2**bits}")
    ax.set_ylabel("振幅")
    ax.set_ylim(-1.0, 1.0)
    ax.legend(loc="lower left", fontsize=8)
    ax.axhline(0, color="0.6", ls="--", lw=0.5)

    snr = compute_snr(sine_demo, q)
    ax.text(0.98, 0.97, f"SNR = {snr:.1f} dB\nq = {q_step:.2e}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

for ax in axes[-1, :]:
    ax.set_xlabel("时间 (s)")

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "quantization_comparison.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
print("原始浮点波形：")
display(ipydisplay.Audio(piano_samples, rate=SAMPLE_RATE))

In [ ]:
print("16-bit 量化（本片段的量化误差很低；是否可闻取决于播放增益与环境）：")
display(ipydisplay.Audio(quantized_versions[16], rate=SAMPLE_RATE))

In [ ]:
print("8-bit 量化（相较 16-bit，量化误差明显增大）：")
display(ipydisplay.Audio(quantized_versions[8], rate=SAMPLE_RATE))

In [ ]:
print("4-bit 量化（在当前片段中，量化误差已明显增大）：")
display(ipydisplay.Audio(quantized_versions[4], rate=SAMPLE_RATE))

### 约一个周期内的量化误差

下面截取 440 Hz 纯音约一个周期，画出 16-bit、8-bit 和 4-bit 量化后的误差序列 $e[n]=x[n]-x_q[n]$。由于 22050/440 不是整数，50 个采样点只近似一个周期。三个子图使用各自的纵轴范围，以便同时观察高位深下的微小误差与低位深下的较大误差。

In [ ]:
# 用约一个周期的纯音比较量化误差；各子图保留独立纵轴尺度
t_demo = np.linspace(0, 1, SAMPLE_RATE, endpoint=False)
sine_demo = 0.9 * np.sin(2 * np.pi * 440 * t_demo)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

# 1 个周期 ≈ 2.27 ms，约 50 个采样点
period_samples = int(SAMPLE_RATE / 440)

for ax, bits in zip(axes, bit_depths):
    q = quantize_float_to_bitdepth(sine_demo, bits)
    q_step = 2.0 / (2 ** bits)

    t_slice = t_demo[:period_samples]
    orig_slice = sine_demo[:period_samples]
    error_slice = orig_slice - q[:period_samples]

    ax.plot(t_slice, error_slice, color="0.15", lw=1.1)
    ax.axhline(0, color="0.65", ls="--", lw=0.6)
    ax.set_title(f"{bits}-bit | 峰值误差 = {np.max(np.abs(error_slice)):.2e}")
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("量化误差")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(-2, 2))

    ax.text(0.98, 0.97, f"量化步长 q = {q_step:.2e}",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="0.9", alpha=0.5))

plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "quantization_error_waveform.png", dpi=600, bbox_inches="tight")
plt.show()

## 6. 响度归一化、削波与相位

前面的量化讨论的是“用多少位记录振幅”。进入实际音频处理前，还需要区分峰值、RMS 幅度、感知响度与相位。

**峰值归一化**只让最大采样点不超过某个上限；**RMS 归一化**把 RMS 振幅调到目标水平；广播与流媒体常用的 LUFS 响度归一化还会引入频率加权和门限，更接近人耳感知。

从整数 PCM 解码的浮点数组通常以约 [-1, 1] 表示满量程，浮点文件本身可以超出该范围。如果增益过大而在导出定点 PCM 或播放链路中进行硬限幅，就会发生**削波**：波峰被压平，并在频谱中制造额外谐波。


In [ ]:
def rms_amplitude(samples):
    """计算整段数组的 RMS 振幅。"""
    samples = np.asarray(samples, dtype=float)
    if samples.size == 0:
        return 0.0
    return float(np.sqrt(np.mean(samples ** 2)))


def peak_normalize(samples, target_peak=0.95):
    """峰值归一化：把最大绝对采样值缩放到 target_peak。"""
    samples = np.asarray(samples, dtype=float)
    peak = float(np.max(np.abs(samples))) if samples.size else 0.0
    return samples.copy() if peak == 0 else samples * (target_peak / peak)


def rms_normalize(samples, target_dbfs=-18.0):
    """RMS 归一化：把 RMS 调到给定 dBFS。"""
    target_rms = 10 ** (target_dbfs / 20.0)
    current_rms = rms_amplitude(samples)
    samples = np.asarray(samples, dtype=float)
    return samples.copy() if current_rms == 0 else samples * (target_rms / current_rms)


# 用纯音演示，避免音乐素材自身动态变化干扰观察。
t_demo = np.linspace(0, 1, SAMPLE_RATE, endpoint=False)
quiet_sine = 0.04 * np.sin(2 * np.pi * 440 * t_demo)
base_sine = 0.45 * np.sin(2 * np.pi * 440 * t_demo)

peak_norm = peak_normalize(quiet_sine, target_peak=0.95)
rms_norm = rms_normalize(quiet_sine, target_dbfs=-18.0)
driven = 3.2 * base_sine
clipped = np.clip(driven, -1.0, 1.0)

fig, axes = plt.subplots(2, 2, figsize=(13, 7.6))
show = slice(0, int(0.010 * SAMPLE_RATE))
time_ms = t_demo[show] * 1000

title_pad = 23
legend_kw = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, frameon=False, fontsize=8, borderaxespad=0.0)

# 1) 峰值归一化：用不同线宽与灰度，并把图例移到绘图区外。
axes[0, 0].plot(time_ms, quiet_sine[show], color="0.62", lw=1.4,
                label=f"原始 peak={np.max(np.abs(quiet_sine)):.2f}")
axes[0, 0].plot(time_ms, peak_norm[show], color="0.05", lw=1.1,
                label="峰值归一化到 0.95")
axes[0, 0].set_title("峰值归一化", pad=title_pad)
axes[0, 0].set_ylabel("振幅")
axes[0, 0].set_ylim(-1.08, 1.08)
axes[0, 0].legend(**legend_kw)
axes[0, 0].axhline(0, color="0.75", lw=0.5)

# 2) RMS 归一化：原始信号更小，归一化后差异不再几乎重叠。
axes[0, 1].plot(time_ms, quiet_sine[show], color="0.62", lw=1.4,
                label=f"原始 RMS={20*np.log10(rms_amplitude(quiet_sine)):.1f} dBFS")
axes[0, 1].plot(time_ms, rms_norm[show], color="0.05", lw=1.1,
                label="RMS 归一化到 -18 dBFS")
axes[0, 1].set_title("RMS 归一化", pad=title_pad)
axes[0, 1].set_ylabel("振幅")
axes[0, 1].set_ylim(-0.22, 0.22)
axes[0, 1].legend(**legend_kw)
axes[0, 1].axhline(0, color="0.75", lw=0.5)

# 3) 削波波形：图例外置，避免遮挡满幅波形。
axes[1, 0].plot(time_ms, driven[show], color="0.62", lw=1.2,
                label="增益后（超出 ±1）")
axes[1, 0].plot(time_ms, clipped[show], color="0.05", lw=1.1,
                label="硬削波后")
axes[1, 0].axhline(1.0, color="0.25", ls="--", lw=0.8)
axes[1, 0].axhline(-1.0, color="0.25", ls="--", lw=0.8)
axes[1, 0].set_title("硬削波", pad=title_pad)
axes[1, 0].set_xlabel("时间 (ms)")
axes[1, 0].set_ylabel("振幅")
axes[1, 0].set_ylim(-1.48, 1.48)
axes[1, 0].legend(**legend_kw)

# 4) 谐波：不用连续曲线，改为离散 stem；两组频率轻微错开，避免完全重叠。
f0 = 440.0
harmonics = np.arange(1, 12)
harmonic_freqs = harmonics * f0
freqs = np.fft.rfftfreq(len(base_sine), 1 / SAMPLE_RATE)


def harmonic_db(samples, harmonic_freqs):
    spectrum = np.abs(np.fft.rfft(samples))
    bin_indices = np.array([np.argmin(np.abs(freqs - f)) for f in harmonic_freqs])
    amplitudes = spectrum[bin_indices]
    return 20 * np.log10(amplitudes / (amplitudes[0] + 1e-12) + 1e-10)

base_harm_db = harmonic_db(base_sine, harmonic_freqs)
clipped_harm_db = harmonic_db(clipped, harmonic_freqs)
floor_db = -90
base_plot = np.maximum(base_harm_db, floor_db)
clipped_plot = np.maximum(clipped_harm_db, floor_db)
offset_hz = 18

axes[1, 1].vlines(harmonic_freqs - offset_hz, floor_db, base_plot, color="0.62", lw=2.0, label="未削波")
axes[1, 1].scatter(harmonic_freqs - offset_hz, base_plot, color="white", edgecolor="0.45", s=28, zorder=3)
axes[1, 1].vlines(harmonic_freqs + offset_hz, floor_db, clipped_plot, color="0.05", lw=2.0, label="削波后")
axes[1, 1].scatter(harmonic_freqs + offset_hz, clipped_plot, color="0.05", s=22, zorder=3)
axes[1, 1].set_xlim(0, harmonic_freqs[-1] + 240)
axes[1, 1].set_ylim(floor_db, 5)
axes[1, 1].set_title("削波谐波失真", pad=title_pad)
axes[1, 1].set_xlabel("谐波频率（以 f₀ 标注）")
axes[1, 1].set_ylabel("相对基频幅度 (dB)")
axes[1, 1].set_xticks(harmonic_freqs[::2])
axes[1, 1].set_xticklabels([f"{h}f₀" for h in harmonics[::2]])
axes[1, 1].legend(**legend_kw)
axes[1, 1].grid(alpha=0.22, axis="y")

for ax in axes[0, :]:
    ax.set_xlabel("时间 (ms)")

plt.subplots_adjust(left=0.07, right=0.98, bottom=0.08, top=0.88, wspace=0.2, hspace=0.5)
plt.savefig(OUTPUT_FIG_DIR / "loudness_normalization_clipping.png", dpi=600, bbox_inches="tight")
plt.show()

print(f"原始峰值：{np.max(np.abs(quiet_sine)):.2f}；峰值归一化后：{np.max(np.abs(peak_norm)):.2f}")
print(f"原始 RMS：{20*np.log10(rms_amplitude(quiet_sine)):.1f} dBFS；RMS 归一化后：{20*np.log10(rms_amplitude(rms_norm)):.1f} dBFS")
print(f"削波前峰值：{np.max(np.abs(driven)):.2f}；削波后峰值：{np.max(np.abs(clipped)):.2f}")


### 相位：同一个频率从哪里开始振动

相位可以理解为周期波形的“横向位置”。两个声波即使频率和振幅完全相同，只要相位不同，波峰和波谷出现的时间就不同。

对单个纯音，整体相位平移通常不改变音高判断；但在多个声波叠加、立体声、麦克风阵列、STFT 重建和源分离中，**相对相位**会直接影响声音是否增强、抵消或产生梳状滤波。


In [ ]:
frequency_hz = 440.0
phase_duration = 2 / frequency_hz
phase_time = np.arange(int(phase_duration * SAMPLE_RATE)) / SAMPLE_RATE

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
title_pad = 23
legend_kw = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False,
                 fontsize=8, borderaxespad=0.0)

phase_styles = [
    (0, "0", "0.10", "-", 1.7),
    (np.pi / 2, "π/2", "0.42", "--", 1.7),
    (np.pi, "π", "0.70", "-.", 1.9),
]

for phase, label, color, linestyle, linewidth in phase_styles:
    wave = np.sin(2 * np.pi * frequency_hz * phase_time + phase)
    axes[0].plot(
        phase_time * 1000,
        wave,
        color=color,
        linestyle=linestyle,
        lw=linewidth,
        label=f"φ={label}",
    )

axes[0].set_title("同频正弦波的相位差", pad=title_pad)
axes[0].set_xlabel("时间 (ms)")
axes[0].set_ylabel("振幅")
axes[0].set_xlim(0, phase_time[-1] * 1000)
axes[0].set_ylim(-1.15, 1.15)
axes[0].legend(ncol=3, **legend_kw)
axes[0].axhline(0, color="0.78", lw=0.6)
axes[0].grid(alpha=0.18, axis="y")

wave_a = np.sin(2 * np.pi * frequency_hz * phase_time)
wave_b = np.sin(2 * np.pi * frequency_hz * phase_time + np.pi)
wave_sum = wave_a + wave_b

axes[1].plot(phase_time * 1000, wave_a, label="A", color="0.18", linestyle="-", lw=1.5)
axes[1].plot(phase_time * 1000, wave_b, label="B: φ=π", color="0.62", linestyle="--", lw=1.7)
axes[1].plot(phase_time * 1000, wave_sum, label="A+B", color="0.02", linestyle="-", lw=2.4)
axes[1].set_title("反相叠加（相位抵消）", pad=title_pad)
axes[1].set_xlabel("时间 (ms)")
axes[1].set_ylabel("振幅")
axes[1].set_xlim(0, phase_time[-1] * 1000)
axes[1].set_ylim(-1.15, 1.15)
axes[1].legend(ncol=3, **legend_kw)
axes[1].axhline(0, color="0.78", lw=0.6, zorder=0)
axes[1].grid(alpha=0.18, axis="y")

plt.subplots_adjust(left=0.07, right=0.98, bottom=0.16, top=0.78, wspace=0.2)
plt.savefig(OUTPUT_FIG_DIR / "phase_basics.png", dpi=600, bbox_inches="tight")
plt.show()


## 7. 真实录音加载与波形可视化

用 `librosa.load` 加载不同素材，观察真实音乐信号的波形特征。
选择四段 3 秒切片：
- 钢琴独奏（乐音，有明确起音与衰减）
- 乐队打击乐（瞬态，尖锐峰值）
- 小提琴独奏（持续音，波形相对密集）
- 人声独唱（浊音/清音交替，波形起伏大）

In [ ]:
# 定义加载函数：统一转单声道、22050 Hz，并截取指定秒数
def load_audio_slice(path, start_sec=0.0, duration_sec=3.0, sample_rate=SAMPLE_RATE):
    """从指定路径加载音频切片，返回采样数组和时间轴"""
    samples, sr = librosa.load(path, sr=sample_rate, mono=True, offset=start_sec, duration=duration_sec)
    time_axis = np.linspace(start_sec, start_sec + len(samples)/sr, len(samples), endpoint=False)
    return time_axis, samples, sr

audio_sources = {
    "钢琴独奏": (DATASET_DIR / "piano_solo.wav", 10.0),      # 从第10秒开始
    "乐队打击乐": (DATASET_DIR / "orch_perc.wav", 2.0),
    "小提琴独奏": (DATASET_DIR / "zhao_violin_wet.wav", 15.0),
    "人声独唱": (DATASET_DIR / "xiaohetang_vox.wav", 30.0),
}

loaded_audio = {}
for label, (path, start) in audio_sources.items():
    if not path.exists():
        warnings.warn(f"跳过缺失文件：{path}")
        continue
    t, samples, sr = load_audio_slice(path, start_sec=start, duration_sec=3.0)
    loaded_audio[label] = (t, samples)
    print(f"{label:25s}: {len(samples):6d} samples @ {sr} Hz | max amp = {np.max(np.abs(samples)):.4f}")

In [ ]:
# 并排波形图
n_sources = len(loaded_audio)
fig, axes = plt.subplots(n_sources, 1, figsize=(12, 2.2 * n_sources), sharex=False)
if n_sources == 1:
    axes = [axes]

for ax, (label, (t, samples)) in zip(axes, loaded_audio.items()):
    librosa.display.waveshow(samples, sr=SAMPLE_RATE, ax=ax, color="0.3", alpha=0.85)
    ax.set_title(label, fontsize=11, loc="left")
    ax.set_xlabel("时间 (s)")
    ax.set_ylabel("振幅")
    max_amp = np.max(np.abs(samples))
    ax.set_ylim(-max_amp * 1.15, max_amp * 1.15)

plt.suptitle("真实录音的波形（3 秒切片）", fontsize=13, y=0.98)
plt.tight_layout()
plt.savefig(OUTPUT_FIG_DIR / "real_audio_waveforms.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# 交互式播放：可试听每一段
for label, (t, samples) in loaded_audio.items():
    print(f"\n{'='*40}\n{label}\n{'='*40}")
    display(ipydisplay.Audio(samples, rate=SAMPLE_RATE))

## 8. 小结与输出清单

1. **合成基础波形**：正弦、方波、锯齿、三角波 @ 440 Hz，保存到 `CODE/chapter05/output_audio/`。
2. **合成噪声**：用固定随机种子生成白噪声与近似粉红噪声；图中 periodogram 围绕理论期望随机起伏。
3. **采样定理演示**：用 3 Hz 余弦展示 10 Hz 采样与 4 Hz 混叠到 1 Hz 的对比。
4. **量化演示**：使用完整的 $2^b$ 个有符号 PCM 码，比较 16-bit / 8-bit / 4-bit 量化与 SNR。
5. **响度、削波与相位**：区分峰值、RMS、感知响度与相位，并展示硬削波产生的谐波。
6. **真实音频加载**：加载并可视化四段项目内录音，固定采样率、截取区间与单声道处理。

下一 Notebook（`02_time_domain_features.ipynb`）将在此基础上计算时域标量特征（AE、RMS、ZCR）。


In [ ]:
# 输出文件清单：只列出本 Notebook 生成的资产
owned_audio = [
    "sine_440hz.wav", "square_440hz.wav", "sawtooth_440hz.wav",
    "triangle_440hz.wav", "white_noise.wav", "pink_noise.wav",
]
owned_figures = [
    "basic_waveforms.png", "waveform_spectra.png", "noise_waveforms_psd.png",
    "aliasing_demo.png", "quantization_comparison.png",
    "quantization_error_waveform.png", "loudness_normalization_clipping.png",
    "phase_basics.png", "real_audio_waveforms.png",
]

print("本 Notebook 生成的音频文件：")
for name in owned_audio:
    f = OUTPUT_AUDIO_DIR / name
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:30s} {size_kb:8.1f} KB")

print("\n本 Notebook 生成的图像文件：")
for name in owned_figures:
    f = OUTPUT_FIG_DIR / name
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s} {size_kb:8.1f} KB")